In [1]:
import numpy as np
import pandas as pd
import  tpqoa
import time
from datetime import datetime, timedelta, timezone
import warnings


warnings.filterwarnings('ignore')

In [99]:
class ConTrader(tpqoa.tpqoa):
    def __init__(self, conf_file, instrument, bar_length, risk_percentage):
        super().__init__(conf_file)
        self.instrument = instrument
        self.bar_length = pd.Timedelta(bar_length)
        self.tick_data = pd.DataFrame()
        self.raw_data = None
        self.last_bar = None
        self.units = 0
        self.signal = {}  #contains strategy_id, timestamp, symbol, direction, entry_type, stop_loss, take_profit, time_stop, ml_probability, ml_regime_label, confidence_score, valid_until, context_tags,reason_code
        self.position = 0
        self.trade_count = 0
        self.current_trade = 0
        self.trade_created_at = 0
        self.summary = self.get_account_summary()
        self.previous_balance = self.summary['balance']
        self.balance = self.summary['balance']
        self.capital = self.summary['balance']
        self.pl = self.summary['pl']
        self.unit = 0
        self.sl_amount = 0
        self.tp_amount = 0
        self.entry = 0
        self.profit = 0
        self.loss = 0
        self.equity = []
        self.leverage = 100
        self.profit_returns = []
        self.loss_returns = []


        self.profits = [] # NEW

        #*****************add strategy-specific attributes here******************
        self.obs = []
        self.active_zones = []
        #************************************************************************

    def get_most_recent(self, days = 5):
        now = datetime.now(timezone.utc).replace(tzinfo=None)
        now = now.replace(second=0, microsecond=0, minute=0)  # floor to the hour
        past = now - timedelta(days = days)
        df = self.get_history(instrument = self.instrument, start = past, end = now,
                               granularity = 'M1', price = "B")
        # df = df.resample(self.bar_length, label = "right").last().dropna().iloc[:-1]

        self.raw_data = df.copy()
        self.dataset_structure()
        for i in range(1, len(self.raw_data)):
            self.analyze_gold_obs(i)

        self.last_bar = self.raw_data.index[-1]

    def dataset_structure(self):
        self.raw_data['body'] = (self.raw_data['c'] - self.raw_data['o']).abs()
        self.raw_data['ATR_14'] = self.raw_data['c'].rolling(14).mean()
        self.raw_data['time'] = self.raw_data.index
        self.raw_data['atr_norm'] = self.raw_data['ATR_14'] / self.raw_data['ATR_14'].rolling(252).mean()

        self.raw_data['vol_regime'] = pd.qcut(
            self.raw_data['atr_norm'],
            q=[0, 0.33, 0.66, 1.0],
            labels=['Low', 'Medium', 'High'])

        self.raw_data['hour'] = self.raw_data.index.hour

        conditions = [
            self.raw_data['hour'].between(0, 8),
            self.raw_data['hour'].between(8, 9),
            self.raw_data['hour'].between(9, 13),
            self.raw_data['hour'].between(13, 17),
            self.raw_data['hour'].between(17, 22),
            self.raw_data['hour'].between(22, 23)
        ]

        choices = [
            'asian',
            'asian/london',
            'london',
            'london/NY',
            'NY',
            'Closing'
        ]

        self.raw_data['sessions'] = np.select(conditions, choices, default='off')


    def on_success(self, time, bid, ask):
        print(self.ticks, end = " ")

        # collect and store tick data
        recent_tick = pd.to_datetime(time).replace(tzinfo=None)
        df = pd.DataFrame({self.instrument:(ask + bid)/2},
                          index = [recent_tick])
        self.tick_data = pd.concat([self.tick_data, df]) # new with pd.concat()

        # if a time longer than the bar_lenght has elapsed between last full bar and the most recent tick
        if recent_tick - self.last_bar >= self.bar_length:
            self.resample_and_join()
            # self.analyze_gold_obs(len(self.raw_data))

            if self.signal :
                self.execute_trades()

    def resample_and_join(self):
        self.raw_data = pd.concat([self.raw_data, self.tick_data.resample(self.bar_length,
                                                                          label="right").last().ffill().iloc[:-1]])
        self.tick_data = self.tick_data.iloc[-1:]
        self.dataset_structure()
        self.last_bar = self.raw_data.index[-1]

    def analyze_gold_obs(self, i, displacement_mult=2.0, forward_window=10):
        """
        df: DataFrame with ['o', 'h', 'l', 'c', 'ATR_14']
        type: either bullish or bearish
        displacement_mult: How much stronger the move must be than the OB candle to count.
        forward_window: How many candles to look ahead for return after a hit.
        """
        curr = self.raw_data.iloc[i]
        prev = self.raw_data.iloc[i-1]

        if self.current_trade != 1:
            if self.position == 1:
                self.current_trade = 1
                self.execute_trades()

        if curr['c'] > curr['o'] and (curr['c'] - curr['o']) > (prev['h'] - prev['l']) * displacement_mult and \
                prev['body'] < prev['ATR_14']:
            if prev['c'] < prev['o']:
                self.active_zones.append({
                    'type': 'Bullish',
                    'top': prev['h'],
                    'bottom': prev['l'],
                    'created_at': i,
                    'created_time': self.raw_data['time'].iloc[i],
                    'status': 'Active'
                })

        for zone in self.active_zones:
            if zone['status'] != 'Active': continue

            # Check for INVALIDATION (Body Close through zone)
            if zone['type'] == 'Bullish' and curr['c'] < zone['bottom']:
                zone['status'] = 'Invalidated'
                continue
            hit = False
            if zone['type'] == 'Bullish':
                # Low crosses mid of order block
                if curr['l'] <= ((zone['top'] + zone['bottom'])/2) and i - zone['created_at'] > 10:
                    hit = True
            if hit:
                self.signal = {
                    'status': 'active',
                    'position': 1,
                    'tp': 100,
                    'sl': 25,
                    'entry_type': "market",
                    'ml_prob':0.6
                }


                self.obs.append({
                    'Type': zone['type'],
                    'Created_At': self.raw_data.index[zone['created_at']],
                    'time': self.raw_data['time'].iloc[i],
                    'vol_regime': self.raw_data['vol_regime'].iloc[i],
                    'sessions': self.raw_data['sessions'].iloc[i],
                    'Hit_At': self.raw_data.index[i],
                    # 'highest_after_hit':self.raw_data['highest'].iloc[i],
                    # 'lowest_after_hit':self.raw_data['lowest'].iloc[i],
                    'Zone_Top': zone['top'],
                    'Zone_Bottom': zone['bottom']
                })
                zone['status'] = 'Mitigated'  # Mark as done

        if i - self.trade_created_at > 20 and self.position == 1:
            print('going neutral')
            self.sell(i, unit=self.unit)
            self.current_trade = 0
            self.position = 0
            if self.balance > self.previous_balance:
                self.profit += 1
                forward_return = ( self.data['c'].iloc[i] - self.entry)/self.entry
                self.profit_returns.append(forward_return)
            elif self.balance < self.previous_balance:
                self.loss += 1
                forward_return = ( self.data['c'].iloc[i] - self.entry)/self.entry
                self.loss_returns.append(forward_return)

            self.previous_balance = self.balance
            self.equity.append({
                'time': self.data.index[i],
                'balance': round(self.balance)
            })

    def execute_trades(self):
        if self.signal:
            if self.signal["position"] == 1:
                if self.position == 0:
                    order = self.create_order(self.instrument, self.units, suppress = True, ret = True)
                    self.report_trade(order, "GOING LONG")  # NEW
                    self.signal['status'] = 'inactive'

                elif self.position == -1:
                    order = self.create_order(self.instrument, self.units * 2, suppress = True, ret = True)
                    self.report_trade(order, "GOING LONG")  # NEW
                    self.signal['status'] = 'inactive'

                self.position = 1


            elif self.signal["position"] == -1:
                if self.position == 0:
                    order = self.create_order(self.instrument, -self.units, suppress = True, ret = True)
                    self.report_trade(order, "GOING SHORT")  # NEW
                    self.signal['status'] = 'inactive'

                elif self.position == 1:
                    order = self.create_order(self.instrument, -self.units * 2, suppress = True, ret = True)
                    self.report_trade(order, "GOING SHORT")  # NEW
                    self.signal['status'] = 'inactive'

                self.position = -1


            elif self.signal["position"] == 0:
                if self.position == -1:
                    order = self.create_order(self.instrument, self.units, suppress = True, ret = True)
                    self.report_trade(order, "GOING NEUTRAL")  # NEW
                    self.signal['status'] = 'inactive'

                elif self.position == 1:
                    order = self.create_order(self.instrument, -self.units, suppress = True, ret = True)
                    self.report_trade(order, "GOING NEUTRAL")  # NEW
                    self.signal['status'] = 'inactive'

                self.position = 0

    def report_trade(self, order, going):  # NEW
        time = order["time"]
        units = order["units"]
        price = order["price"]
        pl = float(order["pl"])
        self.profits.append(pl)
        cumpl = sum(self.profits)
        print("\n" + 100* "-")
        print("{} | {}".format(time, going))
        print("{} | units = {} | price = {} | P&L = {} | Cum P&L = {}".format(time, units, price, pl, cumpl))
        print(100 * "-" + "\n")


In [100]:
trader = ConTrader('../../oanda.cfg', 'XAU_USD', bar_length='1min', risk_percentage=2)

In [101]:
trader.get_most_recent(days=1)

In [ ]:
trader.stream_data(trader.instrument, stop=60)

In [74]:
trader.raw_data

,o,h,l,c,volume,complete,body,ATR_14,time,atr_norm,vol_regime,hour,sessions
time,,,,,,,,,,,,,
2026-04-23 22:04:00,4695.44,4695.65,4694.44,4694.76,36,True,0.68,NaN,2026-04-23 22:04:00,NaN,NaN,22,NY
2026-04-23 22:05:00,4694.76,4697.07,4694.39,4695.80,415,True,1.04,NaN,2026-04-23 22:05:00,NaN,NaN,22,NY
2026-04-23 22:06:00,4695.71,4698.92,4695.39,4698.78,271,True,3.07,NaN,2026-04-23 22:06:00,NaN,NaN,22,NY
2026-04-23 22:07:00,4698.72,4698.72,4696.67,4697.64,238,True,1.08,NaN,2026-04-23 22:07:00,NaN,NaN,22,NY
2026-04-23 22:08:00,4697.84,4698.72,4696.45,4696.95,195,True,0.89,NaN,2026-04-23 22:08:00,NaN,NaN,22,NY
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-24 20:55:00,4706.95,4707.26,4705.85,4706.32,101,True,0.63,4707.395714,2026-04-24 20:55:00,0.997560,Low,20,NY
2026-04-24 20:56:00,4706.29,4706.70,4706.11,4706.29,49,True,0.00,4707.415714,2026-04-24 20:56:00,0.997578,Low,20,NY
2026-04-24 20:57:00,4706.29,4708.21,4705.80,4707.01,170,True,0.72,4707.497857,2026-04-24 20:57:00,0.997609,Low,20,NY


In [28]:
trader.create_order('EUR_USD', 200000, suppress = True, ret = True, sl_distance=0.1, tp_price=1.3)

{'id': '449',
 'time': '2026-04-22T23:21:14.496006731Z',
 'userID': 37974904,
 'accountID': '101-011-37974904-002',
 'batchID': '448',
 'requestID': '115539250145593777',
 'type': 'ORDER_FILL',
 'orderID': '448',
 'instrument': 'EUR_USD',
 'units': '200000.0',
 'gainQuoteHomeConversionFactor': '1.0',
 'lossQuoteHomeConversionFactor': '1.0',
 'price': 1.17064,
 'fullVWAP': 1.17064,
 'fullPrice': {'type': 'PRICE',
  'bids': [{'price': 1.17054, 'liquidity': '500000'},
   {'price': 1.17053, 'liquidity': '500000'},
   {'price': 1.17052, 'liquidity': '2000000'},
   {'price': 1.17051, 'liquidity': '2000000'},
   {'price': 1.1705, 'liquidity': '5000000'},
   {'price': 1.17048, 'liquidity': '10000000'},
   {'price': 1.17045, 'liquidity': '10000000'}],
  'asks': [{'price': 1.17064, 'liquidity': '500000'},
   {'price': 1.17066, 'liquidity': '2500000'},
   {'price': 1.17067, 'liquidity': '2000000'},
   {'price': 1.17068, 'liquidity': '5000000'},
   {'price': 1.17071, 'liquidity': '10000000'},
   {

In [ ]:
trader.get_most_recent()
trader.stream_data(trader.instrument, stop=200)

if trader.position != 0:
    close_order = trader.create_order(trader.instrument, units=-trader.position * trader.units, suppress=True, ret=True)
    trader.report_trade(close_order, "GOING NEUTRAL")
    trader.position = 0

In [28]:
trader.instrument

'XAU_USD'

In [12]:
api = tpqoa.tpqoa('../../oanda.cfg')

In [8]:
api.stream_data("EUR_USD", stop=1)

2026-04-22T23:07:35.877383401Z 1.17051 1.17061
